In [1]:
import time
from datetime import timedelta

import torch
import numpy as np
import uproot

In [ ]:
dataset_info = {
    "W wide mass old 1-2": {
        "file_path": "/eos/home-t/tmlinare/Lund/Lund_tagging/lundtoptagger_data/graphs/ln_kT_cut_None_with_pt_200_with_dsids_mcweights_new_pt_weights_2025-03-07/graphs_Wwidemass_1-2_ln_kT_cut_None_with_pt_200_with_dsids_mcweights_new_pt_weights_2025-03-07",
    },
    "W wide mass new 1-2": {
        "file_path": "/eos/home-t/tmlinare/Lund/Lund_tagging/lundtoptagger_data/graphs/v2.0.1.2/graphs_Wwidemass_1-2_ln_kT_cut_None_with_pt",
        "ROOT_file_path": "/eos/home-t/tmlinare/Lund/Lund_tagging/lundtoptagger_data/graphs/v2.0.1.2/data_Wwidemass_1-2_ln_kT_cut_None.root",
    },
    "W wide mass old 1-50": {
        "file_path": "/eos/home-t/tmlinare/Lund/Lund_tagging/lundtoptagger_data/graphs/ln_kT_cut_None_with_pt_200_with_dsids_mcweights_new_pt_weights_2025-03-07/graphs_Wwidemass_1-50_ln_kT_cut_None_with_pt_200_with_dsids_mcweights_new_pt_weights_2025-03-07",
    },
    "W wide mass new 1-50": {
        "file_path": "/eos/home-t/tmlinare/Lund/Lund_tagging/lundtoptagger_data/graphs/v2.0.1.2/graphs_Wwidemass_1-50_ln_kT_cut_None_with_pt",
        "ROOT_file_path": "/eos/home-t/tmlinare/Lund/Lund_tagging/lundtoptagger_data/graphs/v2.0.1.2/data_Wwidemass_1-50_ln_kT_cut_None.root",
    },
}

In [ ]:
# comparison of old and new graph files
# the new ones have additional selection applied, after applying that new selection on the old ones too, they should match
for dataset_key in dataset_info:
    infile_path = dataset_info[dataset_key]["file_path"]

    print(f"Loading dataset: {dataset_key}")
    print("File path:")
    print(infile_path)
    t_start = time.time()
    dataset_info[dataset_key]["dataset"] = torch.load(infile_path)

    t_elapsed = timedelta(seconds=round(time.time() - t_start))
    print(f"Time taken (hh:mm:ss): {t_elapsed}")
    
    dataset = dataset_info[dataset_key]["dataset"]
    dataset_info[dataset_key]["masses"] = np.array([jet_graph["mass"] for jet_graph in dataset])
    dataset_info[dataset_key]["pts"] = np.array([jet_graph["pt"] for jet_graph in dataset])
    dataset_info[dataset_key]["labels"] = np.array([jet_graph["y"] for jet_graph in dataset])

    masses = dataset_info[dataset_key]["masses"]
    pts = dataset_info[dataset_key]["pts"]
    labels = dataset_info[dataset_key]["labels"]
    selection = (masses>40) & (masses<300) & (pts>200) & (pts<3100) & (labels==1)
    print("Number of jets:", len(dataset_info[dataset_key]["dataset"]))
    print("Number of jets with label 1 passing mass and pt selection:", np.sum(selection))
    print("")

Loading dataset: W wide mass old 1-2
File path:
/eos/home-t/tmlinare/Lund/Lund_tagging/lundtoptagger_data/graphs/ln_kT_cut_None_with_pt_200_with_dsids_mcweights_new_pt_weights_2025-03-07/graphs_Wwidemass_1-2_ln_kT_cut_None_with_pt_200_with_dsids_mcweights_new_pt_weights_2025-03-07
Time taken (hh:mm:ss): 0:00:10
Number of jets: 30405
Number of jets with label 1 passing mass and pt selection: 16230

Loading dataset: W wide mass new 1-2
File path:
/eos/home-t/tmlinare/Lund/Lund_tagging/lundtoptagger_data/graphs/v2.0.1.2/graphs_Wwidemass_1-2_ln_kT_cut_None_with_pt
Time taken (hh:mm:ss): 0:00:04
Number of jets: 16230
Number of jets with label 1 passing mass and pt selection: 16230

Loading dataset: W wide mass old 1-50
File path:
/eos/home-t/tmlinare/Lund/Lund_tagging/lundtoptagger_data/graphs/ln_kT_cut_None_with_pt_200_with_dsids_mcweights_new_pt_weights_2025-03-07/graphs_Wwidemass_1-50_ln_kT_cut_None_with_pt_200_with_dsids_mcweights_new_pt_weights_2025-03-07
Time taken (hh:mm:ss): 0:03:15

In [ ]:
# check that the data saved in graph files matches the data saved in ROOT files
for dataset_key in dataset_info:
    if "ROOT_file_path" in dataset_info[dataset_key]:
        with uproot.open(dataset_info[dataset_key]["ROOT_file_path"]) as f:
            tree = f["FlatSubstructureJetTree"]
            dataset_info[dataset_key]["ROOT_masses"] = tree["fjet_m"].array()
        if np.array_equal(dataset_info[dataset_key]["ROOT_masses"], dataset_info[dataset_key]["masses"]):
            print(f"Mass arrays match for dataset: {dataset_key}")
        else:
            print(f"Mass arrays do NOT match for dataset: {dataset_key}")


Mass arrays match for dataset: W wide mass new 1-2
Mass arrays match for dataset: W wide mass new 1-50
